In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# --- 1. GENERATE SAMPLE DATA ---

np.random.seed(42)
n = 10000

df = pd.DataFrame({
    'CreditScore':     np.random.normal(650, 100, n).clip(350, 850).astype(int),
    'Geography':       np.random.choice(['France', 'Germany', 'Spain'], n, p=[0.5, 0.25, 0.25]),
    'Gender':          np.random.choice(['Male', 'Female'], n),
    'Age':             np.random.normal(45, 15, n).clip(18, 92).astype(int),
    'Tenure':          np.random.randint(0, 10, n),
    'Balance':         np.random.exponential(50000, n).clip(0, 250000).astype(int),
    'NumOfProducts':   np.random.choice([1, 2, 3, 4], n, p=[0.4, 0.4, 0.15, 0.05]),
    'HasCrCard':       np.random.choice([0, 1], n, p=[0.3, 0.7]),
    'IsActiveMember':  np.random.choice([0, 1], n, p=[0.4, 0.6]),
    'EstimatedSalary': np.random.exponential(50000, n).clip(0, 200000).astype(int),
})

churn_prob = (
    (df['Age'] > 50) * 0.2 +
    (df['Balance'] < 10000) * 0.15 +
    (df['NumOfProducts'] == 1) * 0.2 +
    (df['IsActiveMember'] == 0) * 0.2 +
    (df['Geography'] == 'Germany') * 0.1 +
    (df['CreditScore'] < 500) * 0.1
)
df['Exited'] = (np.random.random(n) < churn_prob).astype(int)

print(f"Dataset: {df.shape}")
print(df['Exited'].value_counts())
display(df.head())
print(df.describe())

# --- 2. EXPLORATORY DATA ANALYSIS ---

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

df['Exited'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title('Churn Distribution')
axes[0, 0].set_xticklabels(['Stayed', 'Left'], rotation=0)

df.boxplot(column='Age', by='Exited', ax=axes[0, 1])
axes[0, 1].set_title('Age by Churn Status')

df.boxplot(column='Balance', by='Exited', ax=axes[0, 2])
axes[0, 2].set_title('Balance by Churn Status')

df.groupby('Geography')['Exited'].mean().sort_values().plot(kind='bar', ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Churn Rate by Country')

df.groupby('Gender')['Exited'].mean().plot(kind='bar', color=['#3498db', '#e74c3c'], ax=axes[1, 1])
axes[1, 1].set_title('Churn Rate by Gender')

df.groupby('NumOfProducts')['Exited'].mean().plot(kind='line', marker='o', ax=axes[1, 2])
axes[1, 2].set_title('Churn Rate by Number of Products')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- 3. PREPROCESSING ---

X = df.drop(columns=['Exited']).copy()
y = df['Exited'].copy()

num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(include=['object']).columns

if X[num_cols].isnull().sum().sum() > 0:
    X[num_cols] = SimpleImputer(strategy='median').fit_transform(X[num_cols])
if X[cat_cols].isnull().sum().sum() > 0:
    X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])

for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col])

X[num_cols] = StandardScaler().fit_transform(X[num_cols])

# --- 4. TRAIN / TEST SPLIT ---

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# --- 5. TRAIN THREE MODELS ---

def show_metrics(name, y_true, y_pred, y_proba):
    print(f"\n{name}")
    print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  F1:        {f1_score(y_true, y_pred):.4f}")
    print(f"  ROC-AUC:   {roc_auc_score(y_true, y_proba):.4f}")

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log  = log_reg.predict(X_test)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]
show_metrics("Logistic Regression", y_test, y_pred_log, y_proba_log)

rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
show_metrics("Random Forest", y_test, y_pred_rf, y_proba_rf)

gb = GradientBoostingClassifier(random_state=42, n_estimators=100)
gb.fit(X_train, y_train)
y_pred_gb  = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]
show_metrics("Gradient Boosting", y_test, y_pred_gb, y_proba_gb)

# --- 6. TUNE RANDOM FOREST WITH GRID SEARCH ---

param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
y_pred_best  = best_rf.predict(X_test)
y_proba_best = best_rf.predict_proba(X_test)[:, 1]
show_metrics("Tuned Random Forest", y_test, y_pred_best, y_proba_best)

# --- 7. VISUALIZE RESULTS ---

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for name, proba in [
    ('Logistic Regression', y_proba_log),
    ('Random Forest',       y_proba_rf),
    ('Gradient Boosting',   y_proba_gb),
    ('Tuned RF',            y_proba_best),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0, 0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test, proba):.3f})', linewidth=2)
axes[0, 0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0, 0].set_title('ROC Curves')
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, alpha=0.3)

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
axes[0, 1].set_title('Confusion Matrix — Tuned RF')

feat_imp = pd.DataFrame({'Feature': X.columns, 'Importance': best_rf.feature_importances_})
feat_imp = feat_imp.sort_values('Importance')
axes[1, 0].barh(feat_imp['Feature'], feat_imp['Importance'], color='coral')
axes[1, 0].set_title('Feature Importances')
axes[1, 0].grid(True, alpha=0.3, axis='x')

axes[1, 1].hist(y_proba_best[y_test == 0], bins=30, alpha=0.7, label='Stayed', color='#2ecc71')
axes[1, 1].hist(y_proba_best[y_test == 1], bins=30, alpha=0.7, label='Left',   color='#e74c3c')
axes[1, 1].set_title('Predicted Churn Probability Distribution')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- 8. BUSINESS SUMMARY ---

true_pos  = ((y_pred_best == 1) & (y_test == 1)).sum()
false_pos = ((y_pred_best == 1) & (y_test == 0)).sum()
false_neg = ((y_pred_best == 0) & (y_test == 1)).sum()
savings   = true_pos * 500

print(f"\nCustomers flagged as at-risk: {(y_pred_best == 1).sum()}")
print(f"Actual churners in test set:   {(y_test == 1).sum()}")
print(f"Correctly identified:           {true_pos}")
print(f"False alarms:                   {false_pos}")
print(f"Missed churners:                {false_neg}")
print(f"Estimated savings (€500/client): €{savings:,}")